> **中文阅读版**：本文件是 `10_framework_comparison.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 框架对比：原生 SDK vs CrewAI vs LangGraph

**Docker 镜像**：`ml4t`

notebook 04 到 08 的流水线完全用普通 Python 类端到端构建，没有使用编排框架。这种选择让每次转换都可见，每个产物都可检查，但迫使读者手写编排逻辑。智能体（agentic）框架生态提供了替代方案，可自动完成其中部分工作，代价是采用框架的约定。

本 notebook 用三种风格表达同一个四阶段预测流程，其中只有两种可以比较：

1. **原生 Python SDK**：直接组合本章的 `ResearchAgent` / `DebateAgent` / `SupervisorAgent` 类，形态与 [`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb) 中的 `AIAForecaster` 相同。
2. **LangGraph**：将流水线表示为一个 `StateGraph`，带有类型化状态和四个节点，这些节点委托给原生变体使用的同一组专家类。原生变体和 LangGraph 调用相同的研究、聚合、辩论和监督函数，因此二者的差异仅在编排层，与证据如何被接受无关。
3. **CrewAI**：一份**配置草图**，而非一次运行。CrewAI 的原语（`Agent`、`Task`、`Crew`）按其编写方式展示，代码在此不执行。早先的一轮曾报告该变体的固定概率，但其背后没有任何捕获记录；一个无从检查的数字比没有数字更糟，因此它被删除，留下的只有语法。

因此，比较的是两个共享内部实现的方案之间的**编排层**：各自如何表达四阶段流程、状态存放在哪里，以及中途失败时会发生什么。这是教学实现，而非性能基准——同一个问题的两次运行并不能决定哪个框架预测得更好。

**学习目标**：
- 将同一个专家、聚合、辩论和监督流程表达两次，一次直接组合类，一次作为类型化状态图，并读出图增添了哪些内容
- 说明在每种实现中一次运行的状态存放在哪里，并找到当某个阶段中途失败时你会附加调试器的入口点
- 根据框架接管了什么，以及通过它检查一次运行要付出什么成本，判断一个项目是否应采纳编排框架

**本书参考**：§24.5（工程栈：框架与迁移）。

**前置条件**：[`04_research_agent`](04_research_agent.ipynb) 到 [`08_forecasting_pipeline`](08_forecasting_pipeline.ipynb)，这些 notebook 构建了专家类以及原生和 LangGraph 变体所组合的流水线。

**回放模式 vs 实时模式**：默认情况下（`RUN_LIVE = False`），notebook 回放固定的 2026-06-09 原生和 LangGraph 捕获记录，其中包含真实的模型交互和当天测得的时长。这两个变体在章节较早的聚合下运行，因此其数字出现处标注为 legacy。不会发起任何 API 调用，输出在每台机器上都相同。设置 `RUN_LIVE = True` 可通过本章的 `LLMClient` 针对当前问题运行两个变体，这会花钱且无法复现；使用 `LLM_PROVIDER=mock` 获得确定性客户端。

In [2]:
"""Framework comparison: one four-phase pipeline, native and as a LangGraph."""

import sys

from utils.paths import get_chapter_dir

sys.path.insert(0, str(get_chapter_dir(24)))

import math
import time
from datetime import date
from typing import TypedDict

import polars as pl
from agent_fixtures import get_chapter_contested_question
from agent_observability import TRACES_DIR, RunTrace, replay_llm_calls, trace_llm
from agent_pipeline import logodds_extremize, validate_probabilities
from agent_providers import TokenUsage, create_llm_client
from agent_research import ResearchAgent, format_agent_summary
from agent_schemas import AggregationResult, ForecastQuestion, ForecastResult
from agent_specialists import DebateAgent, SupervisorAgent
from agent_tools import create_search_client

## 设置

将 `RUN_LIVE` 保持为 `False` 会回放固定的 2026-06-09 运行：重新加载保存的原生和 LangGraph 轨迹，不发起任何 API 调用，报告的数字是当天测得的数值。将其设为 `True` 可针对当前问题实时运行两个可执行变体，这会花钱且无法复现。

`N_AGENTS`、`MAX_STEPS`、`DEBATE_ROUNDS` 和 `SUPERVISOR_QUERIES` 配置本章的专家类，原生和 LangGraph 变体都会组合这些类，因此一项更改会以相同方式影响二者。正因如此，比较的是语句数量和状态位置列，而不是两个概率。

轮次预算在这里比在其他地方更重要：一个用完轮次的 agent 完全不会返回概率，而如果一个变体的所有 agent 都如此，看起来像是它失败了，实际失败的是预算。

`LLM_PROVIDER` 为空，因此工厂会选择第一个已设置密钥的提供方。CrewAI 部分不调用模型：它只是打印出来的配置，而 CrewAI 自带的 LiteLLM 层正是本章追踪没有可捕获内容的原因之一。

In [3]:
RUN_LIVE = False
LLM_PROVIDER = ""
N_AGENTS = 3
MAX_STEPS = 5
DEBATE_ROUNDS = 2
SUPERVISOR_QUERIES = 2
PINNED_TRACES = {
    "Native Python": "10_framework_comparison_20260609T150909Z_ba1855847f4c.json",
    "LangGraph": "10_framework_comparison_20260609T151202Z_c8cba3007562.json",
}

## 共享阶段

状态保留所有研究尝试。只有有证据支持的概率进入均值。弃权或某个阶段失败会停止后续模型调用，保留部分证据。置信度是自我报告的，不是校准后的可靠性评分。

In [4]:
class ForecastState(TypedDict, total=False):
    question: ForecastQuestion
    artifacts: list
    aggregation: object
    debate: object
    supervisor: object
    final_probability: float | None
    status: str
    failure_reason: str


def research_node(state, llm, search):
    artifacts = [
        ResearchAgent(llm, search, agent_id=f"agent_{i}", max_steps=MAX_STEPS).run(
            state["question"], market_price=None
        )
        for i in range(N_AGENTS)
    ]
    return {"artifacts": artifacts}


def aggregate_node(state):
    probs = [a.p_yes for a in state["artifacts"] if a.status == "accepted" and a.p_yes is not None]
    if not probs:
        return {"status": "abstained", "failure_reason": "No supported research forecasts"}
    validate_probabilities(probs)
    return {
        "aggregation": AggregationResult("mean", sum(probs) / len(probs), input_probabilities=probs)
    }


def summaries(state):
    return "\n\n".join(
        format_agent_summary(a) for a in state["artifacts"] if a.status == "accepted"
    )


def debate_node(state, llm):
    agent = DebateAgent(llm, max_rounds=DEBATE_ROUNDS)
    try:
        debate = agent.run(
            state["question"].question, summaries(state), state["aggregation"].raw_probability
        )
    finally:
        state["debate"] = agent.artifact
    return {"debate": debate}


def supervise_node(state, llm, search):
    q = state["question"]
    agent = SupervisorAgent(llm, search, max_queries=SUPERVISOR_QUERIES)
    try:
        sup = agent.run(
            q.question,
            summaries(state),
            date.fromisoformat(q.cutoff_date) if q.cutoff_date else None,
        )
    finally:
        state["supervisor"] = agent.artifact
    probability = state["aggregation"].raw_probability
    debate = state["debate"]
    if debate.consensus_reached:
        probability = (debate.bull_final_probability + debate.bear_final_probability) / 2
    if sup.confidence == "high" and sup.p_yes is not None:
        probability = sup.p_yes
    return {"supervisor": sup, "final_probability": logodds_extremize(probability, math.sqrt(3))}


def apply_stage(state, function, *args):
    if state.get("status") != "accepted":
        return state
    try:
        return {**state, **function(state, *args)}
    except Exception as exc:
        return {**state, "status": "failed", "failure_reason": str(exc), "final_probability": None}


def forecast_result(state):
    total = TokenUsage()
    for artifact in [*state.get("artifacts", []), state.get("debate"), state.get("supervisor")]:
        if artifact is not None:
            total += artifact.token_usage
    return ForecastResult(
        question=state["question"],
        agents=state.get("artifacts", []),
        aggregation=state.get("aggregation"),
        debate=state.get("debate"),
        supervisor=state.get("supervisor"),
        final_probability=state.get("final_probability"),
        status=state["status"],
        failure_reason=state.get("failure_reason"),
        total_token_usage=total,
    )

## 原生 Python

普通函数调用持有状态并按顺序执行各阶段。调试和失败处理在 Python 调用栈中仍然可见。

In [5]:
def native_sdk_pipeline(question, llm, search):
    state = {"question": question, "status": "accepted"}
    for function, args in [
        (research_node, (llm, search)),
        (aggregate_node, ()),
        (debate_node, (llm,)),
        (supervise_node, (llm, search)),
    ]:
        state = apply_stage(state, function, *args)
    return forecast_result(state)

## LangGraph

该图声明了相同的四个阶段和边。每个节点委托给相同的函数，并保留相同的产物（artifacts）。本示例演示执行图；持久化/检查点（checkpointing）需要单独配置。

In [6]:
def langgraph_pipeline(question, llm, search):
    from langgraph.graph import END, StateGraph

    graph = StateGraph(ForecastState)
    graph.add_node("research", lambda s: apply_stage(s, research_node, llm, search))
    graph.add_node("aggregate", lambda s: apply_stage(s, aggregate_node))
    graph.add_node("debate", lambda s: apply_stage(s, debate_node, llm))
    graph.add_node("supervise", lambda s: apply_stage(s, supervise_node, llm, search))
    graph.set_entry_point("research")
    graph.add_edge("research", "aggregate")
    graph.add_edge("aggregate", "debate")
    graph.add_edge("debate", "supervise")
    graph.add_edge("supervise", END)
    state = graph.compile().invoke({"question": question, "status": "accepted"})
    return forecast_result(state)

## 用失败夹具（fixture）测试两个编排器

这个明确为合成数据的客户端尝试进行不受支持的预测。两个编排器都不应产生聚合概率或最终概率。本测试不作出任何计时或准确率方面的声明。

In [7]:
class UnsupportedForecastFixture:
    model_name = "synthetic-unsupported-forecast"

    def complete_with_usage(self, messages, json_mode=True):
        return '{"action":"forecast","p_yes":0.9,"rationale":"No evidence"}', TokenUsage()


for pipeline in (native_sdk_pipeline, langgraph_pipeline):
    result = pipeline(
        ForecastQuestion("Synthetic unsupported question?"), UnsupportedForecastFixture(), None
    )
    assert result.status == "abstained" and result.final_probability is None
    assert result.aggregation is None
    print(f"{pipeline.__name__}: {result.status}; {len(result.agents)} attempts retained")

native_sdk_pipeline: abstained; 3 attempts retained


langgraph_pipeline: abstained; 3 attempts retained


## CrewAI 语法示例

CrewAI 使用代理（agents）、任务（tasks）和流程（process）。以下配置仅作为源码展示，并未执行。它没有研究工具或 trace 集成，因此不能作为有证据支持的预测流水线进行比较。在将其输出用作基于研究的预测之前，请实现这些契约。

In [8]:
CREWAI_EXAMPLE = """from crewai import Agent, Task, Crew, Process
analyst = Agent(role="Research analyst", goal="Review supplied evidence", backstory="Forecasting analyst", llm=model)
task = Task(description=question_with_evidence, expected_output="Probability and source-linked rationale", agent=analyst)
crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
# A live application would attach tools, validate outputs and retain actual messages.
"""
print(CREWAI_EXAMPLE)
print("CrewAI: syntax only; no probability, timing or performance measurement")

from crewai import Agent, Task, Crew, Process
analyst = Agent(role="Research analyst", goal="Review supplied evidence", backstory="Forecasting analyst", llm=model)
task = Task(description=question_with_evidence, expected_output="Probability and source-linked rationale", agent=analyst)
crew = Crew(agents=[analyst], tasks=[task], process=Process.sequential)
# A live application would attach tools, validate outputs and retain actual messages.

CrewAI: syntax only; no probability, timing or performance measurement


## 检查实测捕获结果或运行实时版本

默认表格报告原始的实测时长和概率，而不会假装重新运行当前实现。一次全新的实时运行会保留完整的研究、阶段输出、实际消息和计时。不同的随机模型调用并不是对框架效应的受控测试。

In [9]:
rows, records = [], []
if RUN_LIVE:
    llm = create_llm_client(LLM_PROVIDER)
    search = create_search_client()
    question = get_chapter_contested_question()
    for label, pipeline in [
        ("Native Python", native_sdk_pipeline),
        ("LangGraph", langgraph_pipeline),
    ]:
        tracer = trace_llm(llm, label=label)
        started = time.perf_counter()
        result = pipeline(question, tracer, search)
        result.duration_seconds = time.perf_counter() - started
        run = RunTrace.from_result(
            result,
            notebook="10_framework_comparison",
            provider=llm.model_name,
            params={
                "variant": label,
                "n_agents": N_AGENTS,
                "max_steps": MAX_STEPS,
                "debate_rounds": DEBATE_ROUNDS,
                "supervisor_queries": SUPERVISOR_QUERIES,
                "statistical_correction_a": math.sqrt(3),
            },
            llm_calls=tracer.calls,
        )
        print(f"Saved {run.save().name}")
        records.append(run)
        rows.append(
            {
                "variant": label,
                "mode": run.execution_mode,
                "status": result.status,
                "duration_s": result.duration_seconds,
                "final_p": result.final_probability,
            }
        )
else:
    for label, filename in PINNED_TRACES.items():
        run = RunTrace.load(TRACES_DIR / filename)
        records.append(run)
        rows.append(
            {
                "variant": label,
                "mode": "replay",
                "status": "legacy_capture",
                "duration_s": run.params["elapsed_s"],
                "final_p": run.params["result"]["final_p"],
            }
        )
        print(f"{label}: {run.created_at}; {len(run.llm_calls)} actual captured model calls")
print(pl.DataFrame(rows))
print("CrewAI omitted: no inspectable measured capture")

Native Python: 2026-06-09T15:09:09+00:00; 21 actual captured model calls
LangGraph: 2026-06-09T15:12:02+00:00; 19 actual captured model calls
shape: (2, 5)
┌───────────────┬────────┬────────────────┬────────────┬─────────┐
│ variant       ┆ mode   ┆ status         ┆ duration_s ┆ final_p │
│ ---           ┆ ---    ┆ ---            ┆ ---        ┆ ---     │
│ str           ┆ str    ┆ str            ┆ f64        ┆ f64     │
╞═══════════════╪════════╪════════════════╪════════════╪═════════╡
│ Native Python ┆ replay ┆ legacy_capture ┆ 108.0      ┆ 0.3     │
│ LangGraph     ┆ replay ┆ legacy_capture ┆ 102.7      ┆ 0.48    │
└───────────────┴────────┴────────────────┴────────────┴─────────┘
CrewAI omitted: no inspectable measured capture


In [10]:
print(replay_llm_calls(records[0].call_log()[:2], content_chars=600))
print("Full messages remain in the saved JSON; display excerpts are abbreviated.")

  CALL #0 [native]  ·  claude-sonnet-4-20250514  ·  json_mode=True  ·  tokens in/out=254/26
------------------------------------------------------------------------------
  ▸ SYSTEM:
      You are a forecasting agent in a multi-agent forecasting system.
      Your job:
      1) Gather evidence by issuing web/news search queries when needed.
      2) Then produce a binary probability forecast for the question.
      You must follow the action schema exactly and output valid JSON only.
      You must not browse prediction market prices unless they are explicitly
      provided.
  ▸ USER:
      QUESTION:
      Will the Federal Reserve hike rates in 2026?
      MARKET CONTEXT:
      Resolves YES if the upper bound of the federal funds target rate is
      increased at any point between January 1, 2026 and the Fed's December
      2026 meeting (scheduled December 8-9, 2026).
      MARKET IMPLIED PROBABILITY (p_yes):
      0.545
      NEXT ACTION SCHEMA (output JSON only):
      If you need 

语句数量覆盖的是编排层，而不是 agent 类。CrewAI 在编排层最冗长，因为每个 agent 都带有 role / goal / backstory 三元组，每个 task 都带有 description / expected_output / context 列表。LangGraph 的开销在于类型化状态 schema（typed state schema）和四个短小的节点函数；一旦 schema 定义好，每个节点不过少量语句。原生 SDK 则只是一个函数。

两个 `final_p` 值本来就不应相同，它们之间的差距并不能衡量编排：原生实现与 LangGraph 运行相同的 specialist 类、遵循相同的 evidence rule，因此二者数值之所以不同，是因为它们是来自随机模型的两个样本。这正是应当比较结构——编写的语句数量、状态存放位置、调试器挂接位置——而不是预测一致性的理由。

## 对比矩阵

| 维度 | 原生 SDK | CrewAI | LangGraph |
|---|---|---|---|
| **编排形态** | 函数调用栈 | `Crew` 中基于角色的 agent | 带类型化节点的 `StateGraph` |
| **状态位置** | 局部变量 | 由框架串联的 task 输出 | 显式 `TypedDict` |
| **并行性** | 手动（线程 / asyncio） | 按需启用的异步任务 | 条件 / 并行边 |
| **检查点机制** | 自行实现，如 notebook 03 中那样 | 非一等支持 | 一等支持（`BaseCheckpointSaver`） |
| **LLM 层** | 本章的 `LLMClient` | LiteLLM（自动检测 provider） | 透传节点所使用的任何实现 |
| **调试入口** | 抛出异常的那一行 | CrewAI 循环内部 | `app.get_state(thread)` 快照 |
| **完整追踪捕获** | 一个 client 包装器 | 框架回调 / 详细日志 | 一个 client 包装器 |
| **可视化调试** | print / pdb | 详细日志 | 图渲染 + 状态检查 |
| **依赖负担** | 仅 LLM SDK | crewai + langchain 传递依赖 | langgraph + langchain 传递依赖 |
| **学习曲线** | 低（仅 Python） | 中（roles / tasks / processes） | 中高（图 + 状态） |

对第 24 章形态的 pipeline 而言，真正重要的差异可归结为两列：LangGraph 暴露显式的检查点抽象，而原生 Python 不增加任何编排依赖。迁移路径、可观测性方案、框架势头（framework-momentum）权衡，以及关于持久化 / 并行性 / 条件流压力何时足以证明引入框架合理的更详细讨论，放在本书 §24.5，而不是 notebook 中。

## 关键要点

1. **这种形态的 pipeline 从纯 Python 开始。** 它要求调用方编写的编排最少，并把每个失败归到引发它的那一行。这是关于该 pipeline 的结构性证据，而不是对框架的排名。
2. **框架应针对具名压力赢得其位置，而不是凭偏好。** 运行中途的崩溃恢复正是 LangGraph 的检查点抽象所要解决的；具有不同 charter 的具名 persona，正是 CrewAI 的 role 和 goal 槽位所服务的对象。若没有这些压力之一，依赖与间接层就是全部代价。
3. **在采用任何方案之前，先问这次运行的状态存放在哪里。** 它决定运行出错时你能检查什么：可打印的局部变量、框架的 task 输出，或可快照的类型化状态对象。
4. **拥有模型 client 的框架也拥有可观测性。** CrewAI 直接驱动 LiteLLM，因此本章的追踪中没有任何东西能看到它的调用。这就是它在这里以语法而非一次运行的形式出现的原因：没有可供检查的捕获，而边界移动的具体代价就是不可能有捕获。
5. **来自两次运行的两个概率不是基准。** 原生实现与 LangGraph 变体共享内部结构，因此它们数值之间的差异来自采样，而不是编排。比较框架需要结构性度量，这正是语句数量和状态位置列所服务的用途。

**此处构建内容的已知局限性。** 一个问题、每个变体一次运行、一个模型，而且三个变体中只有两个能够执行。语句数量只是调用方编写量的代理指标，并不能说明需要理解的内容有多少：框架可以用语句换取概念。耗时是在同一台机器上对两条不同调用路径的单次测量，采用本章早先的聚合方式取得，而对比矩阵描述的是仍在演进的框架版本。

**下一步**：[`11_research_operator`](11_research_operator.ipynb) 会针对真实案例研究注册表重放一个生产形态的 operator 循环。

**本书**：第 24.5 节涵盖迁移路径、足以证明引入框架合理的持久化、并行性和条件流压力，以及生态系统版本说明。